# Adapter un modèle au corpus kabyle avec Colab
Choisir **Exécution → Modifier le type d’exécution → GPU T4**. Aucun abonnement n’est souscrit par ce notebook.

Modèle candidat : `Qwen/Qwen2.5-0.5B-Instruct`. Sa qualité en kabyle doit être mesurée. On adapte ses traductions avec QLoRA ; ce corpus bilingue ne constitue pas un corpus de conversations natives.

Sur ton PC, après `git pull`, lance `bash scripts/export_chat_dataset.sh`. Importe ensuite le ZIP obtenu. Les traductions liées restent dans le même ensemble pour éviter les fuites entre entraînement et évaluation. Les données non validées restent signalées dans les métadonnées.


In [ ]:
from google.colab import files
from pathlib import Path
import json, zipfile, subprocess, sys, hashlib
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Choisis uniquement l’archive ZIP de l’export.")
archive = Path(next(iter(uploaded)))
dataset = Path('/content/kabyle-dataset')
dataset.mkdir(exist_ok=True)
allowed = {'train.jsonl', 'validation.jsonl', 'test.jsonl', 'provenance.jsonl', 'manifest.json'}
with zipfile.ZipFile(archive) as z:
    if set(z.namelist()) != allowed or sum(x.file_size for x in z.infolist()) > 2_000_000_000:
        raise ValueError('Archive inattendue ou trop volumineuse.')
    z.extractall(dataset)
manifest = json.loads((dataset/'manifest.json').read_text())
print(manifest)


In [ ]:
import re
revision = manifest.get('code_revision', '')
if not re.fullmatch(r'[0-9a-f]{40}', revision):
    raise ValueError('Réexporte le corpus avec le script du dépôt mis à jour : révision Git absente.')
repo = Path('/content/Kabyle')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/liliabkm/Kabyle.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', revision], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', revision], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo/'training/requirements.txt')], check=True)
import torch
assert torch.cuda.is_available(), 'Active le GPU T4 dans les paramètres Colab.'
print(torch.cuda.get_device_name(0))


## Sauvegarde et durée
Le mode `pilot` lance **200 étapes d’optimisation**, pas l’intégralité du corpus. Il sert à vérifier le pipeline. Passe à `full` pour une époque complète ; selon le corpus, cela peut dépasser une session Colab gratuite. Les checkpoints sont sauvegardés toutes les 100 étapes. Google Drive permet de reprendre après interruption, avec le même corpus et le même modèle.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MODE = 'pilot'  # 'pilot' ou 'full'
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
assert MODE in ('pilot', 'full')
run_id = hashlib.sha256(json.dumps(manifest['sha256'], sort_keys=True).encode()).hexdigest()[:12]
output = Path('/content/drive/MyDrive/KabyleChat') / (run_id + '-' + MODE)
command = [sys.executable, str(repo/'training/finetune.py'), '--dataset', str(dataset),
           '--output', str(output), '--base-model', BASE_MODEL,
           '--max-steps', '200' if MODE == 'pilot' else '-1']
if list(output.glob('checkpoint-*')):
    command.append('--resume')
subprocess.run(command, check=True)


## Comparer avant et après
Le score chrF mesure la proximité des traductions de test. Ce n’est pas une mesure de qualité conversationnelle. Fais relire les sorties par un locuteur kabyle, teste des questions inconnues et compare au modèle de base avant de considérer l’adaptation comme une amélioration.


In [ ]:
evaluation = json.loads((output/'evaluation.json').read_text())
print(json.dumps(evaluation, ensure_ascii=False, indent=2))


In [ ]:
import shutil
adapter = output/'adapter'
training = json.loads((adapter/'training_manifest.json').read_text())
assert training['status'] == 'completed'
archive_path = shutil.make_archive('/content/kabyle-chat-adapter', 'zip', root_dir=output, base_dir='adapter')
files.download(archive_path)


## Utiliser sur le site
Sur ton PC, décompresse `kabyle-chat-adapter.zip` dans `artifacts/chat-model/` : le fichier doit se trouver à `artifacts/chat-model/adapter/training_manifest.json`.

Une seule fois : `bash scripts/install_chat_model.sh`.

Puis, dans un terminal séparé : `bash scripts/start_chat_model.sh`. Le premier démarrage télécharge le modèle de base et nécessite Internet. Le serveur reste sur `127.0.0.1:8010`, sans tunnel public. Lance le site normalement avec `bash scripts/start.sh`, puis ouvre **Parler kabyle**. Le CPU peut être lent.
